In [ ]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 18.5 MB/s eta 0:00:00


# Vector Databases

## 1. What is a Database?

A **database** is a system designed to **store, organize, retrieve, and manage data efficiently**.

A database allows applications to perform operations such as:

* **Insert** data
* **Retrieve** data
* **Update** data
* **Delete** data
* **Filter** data
* **Index** data for efficient access

For example, a relational database might store:

```text
ID    Name       Age
─────────────────────
1     Alice      21
2     Bob        24
3     Charlie    22
```

A query such as:

```text
Find the user whose ID is 2
```

is fundamentally a **data retrieval problem**.

Traditional databases are designed primarily around structured data and queries over that data.

---

# 2. What is a Vector?

A **vector** is an ordered collection of numbers.

For example:

```text
[0.12, -0.43, 0.87, 0.21]
```

In NLP, an embedding model converts text into a vector representing aspects of its semantic meaning.

For example:

```text
"How do I renew my license?"
                ↓
        Embedding model
                ↓
[0.12, -0.43, 0.87, ...]
```

The vector itself doesn't contain the original sentence.

It is a **numerical representation** of the information encoded by the embedding model.

---

# 3. Why Do We Need Vector Databases?

Traditional databases are very good at queries such as:

```text
WHERE id = 42
WHERE age > 20
WHERE category = "finance"
```

But semantic retrieval asks a fundamentally different question:

> **Which stored vectors are most similar to this query vector?**

For example:

```text
Query:
"What are the penalties for violating this regulation?"

        ↓ embedding

Query vector
        ↓
[0.21, -0.43, 0.76, ...]
        ↓
Find the most similar stored vectors
```

This requires **vector similarity search**.

A vector database is designed to make this type of retrieval practical at scale.

---

# 4. Definition of a Vector Database

A **vector database** is a database system designed to **store, index, and retrieve vector embeddings efficiently**, usually according to a similarity or distance measure.

It typically manages:

```text
Vectors
+
IDs
+
Metadata
+
Optional original data
+
Vector indexes
```

The important part is that vectors aren't treated merely as arbitrary arrays of numbers.

They become **queryable data**.

---

# 5. What Does a Vector Database Store?

A vector database entry might conceptually look like:

```text
ID:        chunk_1842

Vector:
[0.12, -0.43, 0.87, ...]

Text:
"Applications must be submitted within 30 days..."

Metadata:
    document = "BIS_123"
    page = 42
    section = "Applications"
```

So the database can associate:

```text
vector
   ↓
chunk
   ↓
document
   ↓
metadata
```

This association is extremely important in RAG.

We don't merely want:

```text
similar vector → 👍
```

We want:

```text
similar vector
      ↓
which chunk?
      ↓
which document?
      ↓
which page?
      ↓
what text?
```

That information eventually becomes the **evidence given to the LLM**.

---

# 6. Vector Similarity Search

Suppose the database contains:

```text
Document A → vector A
Document B → vector B
Document C → vector C
```

The user asks:

```text
"What is the minimum age requirement?"
```

The query is converted into:

```text
query → vector Q
```

The vector database then compares:

```text
Q ↔ A
Q ↔ B
Q ↔ C
```

using a distance/similarity measure such as:

* Cosine similarity
* Euclidean (L2) distance
* Inner product

It then returns the nearest/most similar vectors.

---

# 7. Vector Index

Searching every vector individually becomes expensive as the dataset grows.

Suppose we have:

```text
1,000 vectors
```

Brute-force search can compare the query against all 1,000.

But imagine:

```text
100 million vectors
```

Comparing against every vector for every query becomes expensive.

Therefore, vector databases use **vector indexes** to make nearest-neighbor search more efficient.

We've already studied this.

Examples include:

```text
Flat
IVF
HNSW
```

The index organizes the vectors in a way that allows the search algorithm to avoid unnecessary comparisons.

---

# 8. Exact vs Approximate Search

There are two important approaches.

### Exact search

Compare the query against **every vector**.

```text
Query
  ↓
Compare with
  ├── Vector 1
  ├── Vector 2
  ├── Vector 3
  ├── ...
  └── Vector N
```

This gives the true nearest neighbors.

But it can become expensive at large scale.

### Approximate search

Use an index to search only a promising portion of the vector space.

```text
Query
  ↓
Index
  ↓
Promising candidates
  ↓
Nearest neighbors
```

This is generally much faster, but may occasionally miss the mathematically exact nearest neighbor.

This creates the classic tradeoff:

```text
              Search quality
                   ↑
                   │
       Exact       │
                   │
                   │
                   │
                   │
                   └────────────→ Speed
```

In practice, vector systems often accept a tiny amount of recall loss to obtain much better search performance.

---

# 9. Metadata

Vectors alone aren't enough for most applications.

Suppose we have:

```text
Vector → [....]
```

We also want information such as:

```text
document_id
page
section
date
document_type
source
chunk_id
```

This additional information is called **metadata**.

Metadata allows us to perform **filtered retrieval**.

For example:

```text
Find vectors similar to the query

BUT

only from:
document_type = "BIS"
year >= 2024
section = "Safety"
```

Conceptually:

```text
Query
  ↓
Similarity search
  +
Metadata filtering
  ↓
Relevant results
```

This is one of the major advantages of using a database rather than just keeping a raw collection of vectors.

---

# 10. FAISS vs Vector Database

This distinction is worth remembering.

### FAISS

Primarily provides:

```text
Vector indexing
+
Nearest-neighbor search
```

Its core concern is:

> **How do I efficiently search vectors?**

### Vector Database

Provides a broader system around vectors:

```text
Vector storage
+
Vector indexing
+
Similarity search
+
Metadata
+
Filtering
+
IDs
+
Persistence
+
Data management
```

Its concern is closer to:

> **How do I build an application around searchable vector data?**

FAISS can therefore be thought of as a **vector search/indexing building block**, while a vector database is a **larger data-management system**.

The boundary isn't perfectly strict—some vector databases use libraries or algorithms similar to FAISS internally—but the distinction is useful.

---

# 11. Vector Database in a RAG System

A simplified RAG architecture looks like:

```text
                 OFFLINE / INGESTION
                         │
Documents
    ↓
Chunking
    ↓
Embedding Model
    ↓
Vectors
    ↓
┌─────────────────────────────┐
│       Vector Database       │
│                             │
│  vectors                    │
│  metadata                   │
│  chunk IDs                  │
│  vector index               │
└─────────────────────────────┘


                 ONLINE / QUERY
                         │
User Query
    ↓
Embedding Model
    ↓
Query Vector
    ↓
Vector Database
    ↓
Similarity Search
    ↓
Top-k chunks
    ↓
LLM
```

The vector database therefore sits between:

```text
Embedding
    ↓
Retrieval
```

It is part of the **retrieval infrastructure**, not the generation process.

---

# 12. Vector Database ≠ Embedding Model

These are different components.

### Embedding model

Converts:

```text
text → vector
```

Its job is **representation**.

### Vector database

Stores and searches:

```text
vectors → similar vectors
```

Its job is **storage + retrieval**.

So:

```text
"How do I renew my license?"
            ↓
      Embedding model
            ↓
        Query vector
            ↓
      Vector database
            ↓
       Similar chunks
```

Don't collapse these into one concept.

---

# 13. Vector Database ≠ RAG

A vector database is **one component** of a RAG system.

RAG involves much more:

```text
Documents
    ↓
Extraction
    ↓
Chunking
    ↓
Embedding
    ↓
Indexing
    ↓
Retrieval
    ↓
Ranking
    ↓
Context construction
    ↓
LLM
    ↓
Answer
```

The vector database primarily participates around:

```text
Embedding
    ↓
Storage / Indexing
    ↓
Retrieval
```

---

# 14. Why Not Just Use a Normal Database?

You *can* store vectors in a traditional database as arrays or binary data.

The problem is that simply **storing vectors** isn't the same as efficiently **searching vectors**.

A useful vector system needs specialized mechanisms for:

```text
Nearest-neighbor search
        +
Vector indexing
        +
Similarity metrics
        +
Large-scale retrieval
```

That's why vector databases emerged.

However, this leads directly to an important modern development:

> **You don't always need a separate vector database.**

Some traditional relational databases can support vector search through extensions.

The most important example we'll study is:

```text
PostgreSQL
     +
pgvector
     ↓
Relational database
with vector search
```

And **that is our next major step**.

---

# 15. The Mental Model to Remember

If you remember only one diagram from this notebook, make it this:

```text
                 VECTOR DATABASE

       ┌──────────────────────────────┐
       │                              │
       │  Vector                      │
       │    +                         │
       │  Metadata                    │
       │    +                         │
       │  IDs                         │
       │    +                         │
       │  Original / associated data  │
       │    +                         │
       │  Vector Index                │
       │                              │
       └──────────────┬───────────────┘
                      │
                      ↓
              Similarity Search
                      │
                      ↓
                 Top-k results
```

And the conceptual pipeline:

```text
TEXT
 ↓
EMBEDDING
 ↓
VECTOR
 ↓
VECTOR DATABASE
 ↓
SIMILARITY SEARCH
 ↓
RELEVANT CHUNKS
```

### The key distinction

```text
Embedding
    = represent meaning numerically

Vector index
    = organize vectors for efficient search

Vector database
    = store + manage + index + retrieve vector data
```


In [ ]:
chunks = [
    {
        "id": "chunk_1",
        "text": "Students need an identification card to enter the university library.",
        "metadata": {"page": 10, "section": "Library"}
    },
    {
        "id": "chunk_2",
        "text": "The library is open from 8 AM to 10 PM.",
        "metadata": {"page": 11, "section": "Library"}
    },
    {
        "id": "chunk_3",
        "text": "Students must submit assignments before the deadline.",
        "metadata": {"page": 24, "section": "Assignments"}
    }
]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(texts)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)


In [ ]:
print(chunks[0])
# print(embeddings[0])

{'id': 'chunk_1', 'text': 'Students need an identification card to enter the university library.', 'metadata': {'page': 10, 'section': 'Library'}}


In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])

index.add(embeddings)

print(index.ntotal)

3


In [ ]:
print(embeddings.shape)
print(chunks[0])
print(index.ntotal)

(3, 384)
{'id': 'chunk_1', 'text': 'Students need an identification card to enter the university library.', 'metadata': {'page': 10, 'section': 'Library'}}
3


In [ ]:
query = "What do students need to enter the library?"

query_embedding = model.encode([query]).astype("float32")

distances, indices = index.search(query_embedding, k=2)

print(indices)
print(distances)

[[0 2]]
[[0.66027796 1.1683447 ]]


we need a mapping layer

In [ ]:
filtered_chunks = [
    chunk for chunk in chunks
    if chunk["metadata"]["section"] == "Library"
]

for chunk in filtered_chunks:
    print(chunk["id"], "→", chunk["text"])

chunk_1 → Students need an identification card to enter the university library.
chunk_2 → The library is open from 8 AM to 10 PM.


We haven't actually combined this with vector search yet.

We've only demonstrated:

```text
Metadata filtering
```

separately.

The real retrieval problem is:

> **Find the most similar vectors, but only among records satisfying some metadata condition.**

For example:

```text
Query:
"What are the library hours?"

Filter:
section = "Library"

                ↓

        Candidate chunks
        ┌───────────────┐
        │ chunk_1       │
        │ chunk_2       │
        └───────────────┘
                ↓
         Vector search
                ↓
          Top-k results
```

That's the next little experiment.



In [ ]:
# Create the query
query = "What are the library hours?"

query_embedding = model.encode([query]).astype("float32")

In [ ]:
# Filter the chunks
filtered_chunks = [
    chunk for chunk in chunks
    if chunk["metadata"]["section"] == "Library"
]

for chunk in filtered_chunks:
    print(chunk["id"], "→", chunk["text"])

chunk_1 → Students need an identification card to enter the university library.
chunk_2 → The library is open from 8 AM to 10 PM.


Remember: our original embeddings array is still:

    embedding[0] → chunk_1
    embedding[1] → chunk_2
    embedding[2] → chunk_3

So we need to preserve that relationship.

In [ ]:
filtered_indices = [
    i for i, chunk in enumerate(chunks)
    if chunk["metadata"]["section"] == "Library"
]

filtered_embeddings = embeddings[filtered_indices]

In [ ]:
# Now we search only these vectors
filtered_index = faiss.IndexFlatL2(embeddings.shape[1])

filtered_index.add(filtered_embeddings)

distances, indices = filtered_index.search(
    query_embedding,
    k=2
)

print(indices)
print(distances)

[[1 0]]
[[0.51185524 1.3125967 ]]


In [ ]:
# But now we can translate them back to text
for i, distance in zip(indices[0], distances[0]):
    original_index = filtered_indices[i]

    print(chunks[original_index]["id"])
    print(chunks[original_index]["text"])
    print("Distance:", distance)
    print()

chunk_2
The library is open from 8 AM to 10 PM.
Distance: 0.51185524

chunk_1
Students need an identification card to enter the university library.
Distance: 1.3125967



Now the important question: why use a vector database?

Look at what we had to manually do:

    chunks
        ↓
    embeddings
        ↓
    FAISS index
        ↓
    maintain index ↔ chunk mapping
        ↓
    manually filter metadata
        ↓
    create another FAISS index
        ↓
    search
        ↓
    map results back